# MedNorm-VI E4 PhoBERT W2NER Training

Intended environment: Google Colab with Google Drive mounted. This notebook runs E4 smoke training by default and can run full training only after the explicit authorization string is set. Artifacts are written under `/content/drive/MyDrive/MedNorm-VI/artifacts/`; model caches stay outside artifact directories. It never runs organizer inference, never writes `output.zip`, and never accesses internal_test.

## Audit 0038 input contract

The W2NER relation grid is indexed by **atomic original-text words**, not by VnCoreNLP segmented model words. VnCoreNLP output is used only as PhoBERT input. A real governed entity proved the two coordinate systems must be decoupled:

```text
vimedner:train:train-000054   (validation split, row 4)
gold SYMPTOM 85:102 "rối loạn nhịp tim"
VnCoreNLP model word "gây_rối" spans 81:88, so the gold start 85 fell INSIDE one model word
```

Atomic words give `gây 81:84` and `rối 85:88`, so the entity aligns to `rối | loạn | nhịp | tim`. An Audit-0037 checkpoint describes a different input space and is rejected, not resumed.

## Run-all order (one fresh `Runtime -> Run all`)

1. mount Drive; 2. clone/update repository; 3. repo root + `PYTHONPATH`; 4. resolve governed corpus by authoritative SHA-256; 5. resolve immutable model/tokenizer revisions; 6. acquire **tokenizer only**; 7. atomic original-word surfaces; 8. VnCoreNLP model-word surfaces; 9. validate the atomic/model-word projection; 10. complete train + validation corpus preflight; 11. print and save the diagnostic summary; 12. **only if preflight passes**, acquire the 1.48 GB encoder; 13. smoke/full training; 14. save, reload, hash and validate the artifact.

The large encoder is never downloaded or instantiated before the full alignment preflight passes. VnCoreNLP is initialized once per process and is not reset when a cell is rerun.


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = os.environ.get("MEDNORM_REPO_URL", "https://github.com/vquclinh/MedNorm-VI")
REPO_REF = os.environ.get("MEDNORM_REPO_REF", "main")
CORPUS_DIR = DRIVE_ROOT / "data"
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_smoke_v1"
FULL_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_full_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
RUN_SMOKE_TRAINING = True
RUN_FULL_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
RESUME_FROM_FULL_CHECKPOINT = False
SEED = 20260727
SMOKE_EPOCHS = 1
FULL_EPOCHS = 12
EFFECTIVE_BATCH_SIZE = 8
MAX_WORDS = 256
MAX_MODEL_TOKENS = 512
SMOKE_ROWS = 8
BOOTSTRAP_DEPENDENCIES = (
    "transformers>=4.41",
    "huggingface_hub>=0.23",
    "safetensors>=0.4",
    "py_vncorenlp>=0.1.4",
)

try:
    from google.colab import drive, userdata
except ModuleNotFoundError:
    drive = None
    userdata = None

if drive is None:
    raise RuntimeError("This training notebook is intended for Colab; google.colab.drive is unavailable")
drive.mount("/content/drive")
if not DRIVE_ROOT.exists():
    raise RuntimeError(f"Drive root is unavailable after mount: {DRIVE_ROOT}")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "--prune"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", RESOLVED_COMMIT):
    raise AssertionError("repository commit must be a 40-hex SHA")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *BOOTSTRAP_DEPENDENCIES], check=True)
os.chdir(REPO_DIR)
repo_src = (REPO_DIR / "src").resolve()
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))
os.environ["PYTHONPATH"] = str(repo_src) + os.pathsep + os.environ.get("PYTHONPATH", "")

import mednorm_vi  # noqa: E402
module_path = Path(mednorm_vi.__file__).resolve()
if repo_src not in module_path.parents:
    raise RuntimeError(f"mednorm_vi imported from {module_path}, not from cloned repo {repo_src}")

random.seed(SEED)
LOCAL_PROTOCOL_ASSERTION = dict(internal_test_accessed=False)
print(json.dumps({
    "stage": "bootstrap",
    "repository_commit": RESOLVED_COMMIT,
    "repo_dir": str(REPO_DIR),
    "mednorm_vi_import": str(module_path),
}, indent=2, sort_keys=True))

In [ ]:
from mednorm_vi.mention_factory.w2ner import (
    ATOMIC_WORD_POLICY_VERSION,
    EntitySpan,
    build_relation_grid_head,
    decode_w2ner_grid,
    tokenize_atomic_words,
)
from mednorm_vi.training.phase2.artifacts import STATUS_FULLY_TRAINED, STATUS_SMOKE_EXECUTED, validate_e4_artifact
from mednorm_vi.training.phase2.common import canonical_json_sha256, sha256_file
from mednorm_vi.training.phase2.e4_alignment_diagnostic import run_alignment_diagnostic
from mednorm_vi.training.phase2.e4_w2ner_training import (
    ATOMIC_PROJECTION_VERSION,
    E4_INPUT_CONTRACT_VERSION,
    E4_FULL_AUTHORIZATION,
    E4_GOVERNED_TRAIN_SHA256,
    E4_GOVERNED_VALIDATION_SHA256,
    E4_MODEL_ID,
    assert_full_not_initialized_from_smoke,
    build_e4_manifest,
    build_e4_resolved_config,
    atomic_relation_head_input_dim,
    build_atomic_projection,
    build_w2ner_batch_contract_from_segmented_words,
    decode_w2ner_logits,
    e4_checkpoint_payload,
    prepare_phobert_word_inputs,
    project_to_atomic_word_embeddings,
    reject_incompatible_e4_checkpoint,
    resolve_e4_governed_splits,
    validate_phobert_encoder_load_report,
)
from mednorm_vi.training.phobert_alignment import (
    map_segmented_words,
    resolve_segmented_text,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)

EXPECTED_CORPUS_HASHES = {
    "train": E4_GOVERNED_TRAIN_SHA256,
    "validation": E4_GOVERNED_VALIDATION_SHA256,
}

def validate_corpus_hashes(CORPUS_DIR: Path) -> dict[str, str]:
    global split_resolutions
    search_roots = (
        CORPUS_DIR,
        CORPUS_DIR / "processed",
        DRIVE_ROOT / "data" / "derived",
        REPO_DIR / "data",
    )
    split_resolutions = resolve_e4_governed_splits(search_roots)
    observed = {name: resolution.sha256 for name, resolution in split_resolutions.items()}
    if observed != EXPECTED_CORPUS_HASHES:
        raise AssertionError("governed corpus hashes do not match authoritative E4 splits")
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)
TRAIN_SPLIT_PATH = split_resolutions["train"].path
VALIDATION_SPLIT_PATH = split_resolutions["validation"].path
print(json.dumps({
    "stage": "corpus_resolution",
    "corpus_hashes": corpus_hashes,
    "train_path": str(TRAIN_SPLIT_PATH),
    "validation_path": str(VALIDATION_SPLIT_PATH),
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

In [ ]:
IMMUTABLE_REVISION_RE = re.compile(r"[0-9a-f]{40}")

def optional_hf_token() -> str | None:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    if userdata is None:
        return None
    try:
        secret = userdata.get("HF_TOKEN")
    except Exception:
        return None
    return str(secret).strip() or None

def resolve_hf_revision(model_id: str, *, env_var: str, requested_revision: str = "main") -> str:
    env_revision = os.environ.get(env_var, "").strip()
    if env_revision:
        if not IMMUTABLE_REVISION_RE.fullmatch(env_revision):
            raise SystemExit(f"{env_var} must be an immutable 40-hex revision, not {env_revision!r}")
        return env_revision
    from huggingface_hub import HfApi
    info = HfApi(token=optional_hf_token()).model_info(model_id, revision=requested_revision)
    resolved = str(info.sha)
    if not IMMUTABLE_REVISION_RE.fullmatch(resolved):
        raise SystemExit(f"Hugging Face did not resolve {model_id} to an immutable 40-hex revision")
    return resolved

PINNED_MODEL_REVISION = resolve_hf_revision(E4_MODEL_ID, env_var="MEDNORM_E4_MODEL_REVISION")
PINNED_TOKENIZER_REVISION = os.environ.get("MEDNORM_E4_TOKENIZER_REVISION", "").strip() or PINNED_MODEL_REVISION

def require_resolved_revision(value: str, field_name: str) -> None:
    if not IMMUTABLE_REVISION_RE.fullmatch(value):
        raise SystemExit(f"{field_name} must be resolved before the revision gate")

require_resolved_revision(PINNED_MODEL_REVISION, "PINNED_MODEL_REVISION")
require_resolved_revision(PINNED_TOKENIZER_REVISION, "PINNED_TOKENIZER_REVISION")
print(json.dumps({
    "stage": "revision_resolution",
    "model_id": E4_MODEL_ID,
    "model_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": PINNED_TOKENIZER_REVISION,
    "hf_token_present": optional_hf_token() is not None,
}, indent=2, sort_keys=True))

In [ ]:
assert_full_not_initialized_from_smoke(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
)
if RUN_FULL_TRAINING and CONFIRM_FULL != E4_FULL_AUTHORIZATION:
    raise SystemExit("E4 full training requires explicit operator authorization")
if not (RUN_SMOKE_TRAINING or RUN_FULL_TRAINING):
    raise SystemExit("Run all defaults to smoke; set RUN_SMOKE_TRAINING=True or RUN_FULL_TRAINING=True")

OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
mode = "full" if RUN_FULL_TRAINING else "smoke"
resolved_config = build_e4_resolved_config(
    mode=mode,
    model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_TOKENIZER_REVISION,
    seed=SEED,
    max_words=MAX_WORDS,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
)
(OUTPUT_DIR / "resolved_config.json").write_text(json.dumps(resolved_config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
CONFIG_SHA256 = canonical_json_sha256(resolved_config)
print(json.dumps({
    "stage": "run_gate",
    "mode": mode,
    "output_dir": str(OUTPUT_DIR),
    "config_sha256": CONFIG_SHA256,
    "full_authorized": RUN_FULL_TRAINING and CONFIRM_FULL == E4_FULL_AUTHORIZATION,
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

In [ ]:
# ---------------------------------------------------------------------------
# STEP 6: acquire the TOKENIZER ONLY. The 1.48 GB encoder is not touched here.
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer

USE_FAST_TOKENIZER = os.environ.get("MEDNORM_E4_USE_FAST_TOKENIZER", "0") == "1"
tokenizer = AutoTokenizer.from_pretrained(
    E4_MODEL_ID,
    revision=PINNED_TOKENIZER_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=USE_FAST_TOKENIZER,
    token=optional_hf_token(),
    local_files_only=False,
)
tokenizer_report = {
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "slow_path_required_for_official_phobert": not bool(getattr(tokenizer, "is_fast", False)),
}

# VnCoreNLP SINGLETON. py_vncorenlp starts a JVM and chdirs into save_dir; starting
# a second one in the same process raises. The annotator is cached in globals() so a
# cell rerun reuses it instead of re-initializing, and the working directory is
# restored immediately.
if globals().get("VNCORENLP_ANNOTATOR") is None:
    import py_vncorenlp
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _cwd_before_jvm = Path.cwd()
    try:
        VNCORENLP_ANNOTATOR = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    finally:
        os.chdir(_cwd_before_jvm)
    VNCORENLP_INITIALIZED_ONCE = True
else:
    VNCORENLP_INITIALIZED_ONCE = False

def segment_with_vncorenlp(text: str) -> str:
    segments = VNCORENLP_ANNOTATOR.word_segment(text)
    if not segments:
        raise RuntimeError("VnCoreNLP returned no segments")
    return " ".join(segments)

print(json.dumps({
    "stage": "tokenizer_and_segmenter_acquisition",
    **tokenizer_report,
    "vncorenlp_initialized_this_run": VNCORENLP_INITIALIZED_ONCE,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# STEPS 7-9: atomic original-word surface, VnCoreNLP model-word surface, and the
# projection between them. Still no encoder.
# ---------------------------------------------------------------------------
def entity_from_row(ent: dict) -> EntitySpan:
    entity_type = ent.get("target_type") or ent.get("type") or ent.get("label")
    if entity_type is None:
        raise RuntimeError("governed entity has no organizer type field")
    return EntitySpan(int(ent["start"]), int(ent["end"]), str(entity_type), str(ent["text"]))

def build_surfaces(text: str):
    """(atomic grid words, VnCoreNLP model words) for one governed example."""
    atomic_words = tokenize_atomic_words(text)                       # STEP 7
    segmented_text, segmentation_source = resolve_segmented_text(text, segment_with_vncorenlp)
    model_words = map_segmented_words(text, segmented_text_to_words(segmented_text))  # STEP 8
    return atomic_words, model_words, segmentation_source

# STEP 9: the projection must hold on the exact example that failed in Colab.
PROBE_TEXT = (
    "tìm kiếm các dấu hiệu của các bệnh khác , chẳng hạn như bệnh tuyến giáp , "
    "có thể gây rối loạn nhịp tim ."
)
probe_atomic, probe_model, _probe_source = build_surfaces(PROBE_TEXT)
probe_entity = EntitySpan(85, 102, "SYMPTOM", PROBE_TEXT[85:102])
if probe_entity.text != "rối loạn nhịp tim":
    raise AssertionError("probe entity text drifted from the audited Colab failure")
probe_contract = build_w2ner_batch_contract_from_segmented_words(
    "probe", PROBE_TEXT, (probe_entity,), probe_model, max_words=MAX_WORDS,
)
probe_encoding = prepare_phobert_word_inputs(tokenizer, probe_contract.segmented_words, max_length=MAX_MODEL_TOKENS)
probe_projection = build_atomic_projection(
    PROBE_TEXT, probe_contract.segmented_words, probe_encoding, atomic_words=probe_contract.atomic_words,
)
probe_decoded = {(s.start, s.end, s.entity_type) for s in decode_w2ner_grid(probe_contract.grid)}
if (85, 102, "SYMPTOM") not in probe_decoded:
    raise AssertionError("atomic grid failed to represent the audited governed entity")
probe_features = {
    word.text: probe_projection.atomic_features[index]
    for index, word in enumerate(probe_projection.atomic_words)
    if word.text in {"gây", "rối"}
}
if len(probe_features) != 2 or len(set(probe_features.values())) != 2:
    raise AssertionError("atomic words under one merged model token must stay distinguishable")
print(json.dumps({
    "stage": "surface_and_projection_validation",
    "atomic_word_count": len(probe_atomic),
    "model_word_count": len(probe_model),
    "audited_entity_representable": True,
    "merged_model_words": probe_projection.merged_model_word_count,
    "multi_model_atomic_words": probe_projection.multi_model_word_atomic_count,
    "gay_roi_features": {key: list(value) for key, value in sorted(probe_features.items())},
    "input_contract_version": E4_INPUT_CONTRACT_VERSION,
    "atomic_projection_version": ATOMIC_PROJECTION_VERSION,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# STEPS 10-11: complete governed train + validation alignment preflight. The
# encoder is acquired only if this passes.
# ---------------------------------------------------------------------------
alignment_diagnostic = run_alignment_diagnostic(
    {"train": TRAIN_SPLIT_PATH, "validation": VALIDATION_SPLIT_PATH},
    segmenter=segment_with_vncorenlp,
    tokenizer=tokenizer,
    max_words=MAX_WORDS,
    max_model_tokens=MAX_MODEL_TOKENS,
)
DIAGNOSTIC_SHA256 = alignment_diagnostic.write(OUTPUT_DIR / "e4_alignment_diagnostic.json")
diagnostic_summary = alignment_diagnostic.summary()
print(json.dumps({
    "stage": "full_corpus_alignment_preflight",
    "diagnostic_sha256": DIAGNOSTIC_SHA256,
    "summary": diagnostic_summary,
    "internal_test_accessed": False,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))

# Governed policy: every entity must align to atomic word boundaries, with no
# snapping, no trimming and no silent exclusion.
if not alignment_diagnostic.passed:
    raise AssertionError(
        "E4 alignment preflight failed: "
        f"{alignment_diagnostic.unalignable_after_atomic} unalignable entities, "
        f"{alignment_diagnostic.silent_exclusions} exclusions, "
        f"{alignment_diagnostic.projection_violations} projection violations"
    )
PREFLIGHT_PASSED = True

# Governed W2NER contracts, built on the atomic grid surface.
def load_governed_w2ner_contracts(split_path: Path, max_rows: int | None = None):
    contracts = []
    alignment_reports = []
    with Path(split_path).open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if max_rows is not None and index >= max_rows:
                break
            row = json.loads(line)
            text = str(row["text"])
            entities = tuple(entity_from_row(ent) for ent in row.get("entities", []))
            _atomic, model_words, segmentation_source = build_surfaces(text)
            verify_tokenizer_equivalence(model_words, tokenizer)
            contract = build_w2ner_batch_contract_from_segmented_words(
                str(row.get("example_id", row.get("id", index))),
                text,
                entities,
                model_words,
                max_words=MAX_WORDS,
            )
            encoding = prepare_phobert_word_inputs(tokenizer, contract.segmented_words, max_length=MAX_MODEL_TOKENS)
            if "offset_mapping" in encoding.model_inputs:
                raise RuntimeError("offset_mapping leaked into encoder inputs")
            projection = build_atomic_projection(
                text, contract.segmented_words, encoding, atomic_words=contract.atomic_words,
            )
            if projection.atomic_word_count != contract.word_count:
                raise RuntimeError("projection and W2NER grid disagree on the atomic word count")
            contracts.append(contract)
            alignment_reports.append({
                "row_index": index,
                "segmentation_source": segmentation_source,
                "atomic_word_count": contract.word_count,
                "model_word_count": len(contract.segmented_words),
                "encoded_tokens": len(encoding.model_inputs["input_ids"]),
                "tokenizer_is_fast": encoding.tokenizer_is_fast,
                "consumed_offset_mapping": encoding.consumed_offset_mapping,
            })
    if not contracts:
        raise RuntimeError("no W2NER contracts were loaded")
    return contracts, alignment_reports

row_limit = None if RUN_FULL_TRAINING else SMOKE_ROWS
train_contracts, train_alignment_reports = load_governed_w2ner_contracts(TRAIN_SPLIT_PATH, max_rows=row_limit)
validation_contracts, validation_alignment_reports = load_governed_w2ner_contracts(VALIDATION_SPLIT_PATH, max_rows=row_limit)
grid_target_statistics = {
    "train_contracts": len(train_contracts),
    "validation_contracts": len(validation_contracts),
    "max_atomic_words": max(item.word_count for item in train_contracts + validation_contracts),
    "label_count": train_contracts[0].label_count,
    "train_hash": corpus_hashes["train"],
    "validation_hash": corpus_hashes["validation"],
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "input_contract_version": E4_INPUT_CONTRACT_VERSION,
    "grid_word_surface": ATOMIC_WORD_POLICY_VERSION,
    "diagnostic_sha256": DIAGNOSTIC_SHA256,
}
(OUTPUT_DIR / "grid_target_statistics.json").write_text(json.dumps(grid_target_statistics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "stage": "w2ner_contract_construction",
    "grid_target_statistics": grid_target_statistics,
    "sample_alignment": (train_alignment_reports + validation_alignment_reports)[:3],
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))


In [ ]:
def _gold_exact_set(item):
    return {(span.start, span.end, span.entity_type) for span in decode_w2ner_grid(item.grid)}

def _atomic_word_embeddings(base_model, tokenizer, item, device):
    """Encode one example and project PhoBERT states onto ATOMIC grid words."""
    import torch

    encoding = prepare_phobert_word_inputs(tokenizer, item.segmented_words, max_length=MAX_MODEL_TOKENS)
    if "offset_mapping" in encoding.model_inputs:
        raise RuntimeError("offset_mapping leaked into encoder inputs")
    projection = build_atomic_projection(
        item.grid.original_text, item.segmented_words, encoding, atomic_words=item.atomic_words,
    )
    if projection.atomic_word_count != item.word_count:
        raise RuntimeError("projection and W2NER grid disagree on the atomic word count")
    model_inputs = {
        key: torch.tensor([value], dtype=torch.long, device=device)
        for key, value in encoding.model_inputs.items()
    }
    outputs = base_model(**model_inputs)
    return project_to_atomic_word_embeddings(outputs.last_hidden_state[0], projection)

def _predict_exact_set(base_model, head, tokenizer, item, device):
    import torch
    word_embeddings = _atomic_word_embeddings(base_model, tokenizer, item, device)
    pair_mask = torch.tensor([item.grid.pair_mask], dtype=torch.bool, device=device)
    logits = head(word_embeddings, pair_mask)[0].detach().cpu().tolist()
    return set(decode_w2ner_logits(item, logits))

def evaluate_w2ner_validation(base_model, head, tokenizer, validation_contracts, device) -> dict[str, float | int | bool]:
    import torch
    base_model.eval()
    head.eval()
    true_positive = 0
    predicted_total = 0
    gold_total = 0
    with torch.no_grad():
        for item in validation_contracts:
            predicted = _predict_exact_set(base_model, head, tokenizer, item, device)
            gold = _gold_exact_set(item)
            true_positive += len(predicted & gold)
            predicted_total += len(predicted)
            gold_total += len(gold)
    precision = true_positive / predicted_total if predicted_total else 0.0
    recall = true_positive / gold_total if gold_total else 0.0
    exact_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "validation_exact_precision": precision,
        "validation_exact_recall": recall,
        "validation_exact_f1": exact_f1,
        "validation_true_positive": true_positive,
        "validation_predicted_total": predicted_total,
        "validation_gold_total": gold_total,
        "internal_test_accessed": False,
    }

def checkpoint_state_payload(*, base_model, head, epoch: int, optimizer_steps: int, parameter_count: int, mode: str):
    payload = e4_checkpoint_payload(
        mode=mode,
        config_sha256=CONFIG_SHA256,
        model_revision=PINNED_MODEL_REVISION,
        tokenizer_revision=PINNED_TOKENIZER_REVISION,
        parameter_count=parameter_count,
    )
    payload["epoch"] = epoch
    payload["optimizer_steps"] = optimizer_steps
    payload["model_state"] = {
        "base_model": base_model.state_dict(),
        "w2ner_head": head.state_dict(),
    }
    return payload

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    import torch
    if not path.is_file():
        raise AssertionError(f"missing checkpoint: {path}")
    if sha256_file(path) != expected_sha256:
        raise AssertionError(f"checkpoint hash changed after save: {path}")
    payload = torch.load(path, map_location="cpu")
    required = {"checkpoint_schema_version", "expert_id", "mode", "config_sha256", "model_revision", "model_state"}
    missing = required - set(payload)
    if missing:
        raise AssertionError(f"checkpoint payload missing keys: {sorted(missing)}")
    if payload["config_sha256"] != CONFIG_SHA256:
        raise AssertionError("checkpoint config hash does not match resolved_config.json")
    # An Audit-0037 checkpoint describes the segmented-model-word grid and must
    # never be accepted or resumed under the atomic-grid contract.
    reject_incompatible_e4_checkpoint(payload)

def run_training(contracts, validation_contracts, *, mode: str, epochs: int):
    import torch
    from torch import nn
    from transformers import AutoModel

    # STEP 12: the 1.48 GB encoder is acquired ONLY after the full-corpus alignment
    # preflight passed. Downloading it earlier wastes a Colab session on a run that
    # cannot build its training targets.
    if not globals().get("PREFLIGHT_PASSED", False):
        raise RuntimeError("refusing to acquire the PhoBERT encoder before the alignment preflight passes")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model, loading_info = AutoModel.from_pretrained(
        E4_MODEL_ID,
        revision=PINNED_MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        token=optional_hf_token(),
        local_files_only=False,
        use_safetensors=True,
        output_loading_info=True,
    )
    load_report = validate_phobert_encoder_load_report(
        missing_keys=tuple(loading_info.get("missing_keys", ())),
        unexpected_keys=tuple(loading_info.get("unexpected_keys", ())),
    )
    base_model.to(device)
    head = build_relation_grid_head(
        atomic_relation_head_input_dim(base_model.config.hidden_size),
        contracts[0].label_count,
    ).to(device)
    parameter_count = sum(parameter.numel() for parameter in base_model.parameters()) + sum(parameter.numel() for parameter in head.parameters())
    optimizer = torch.optim.AdamW(list(base_model.parameters()) + list(head.parameters()), lr=2e-5)
    history_path = OUTPUT_DIR / "logs" / "training_history.jsonl"
    if not RESUME_FROM_FULL_CHECKPOINT:
        history_path.write_text("", encoding="utf-8")
    best_metric = -1.0
    optimizer_steps_total = 0
    best_payload = None
    latest_payload = None
    for epoch in range(1, epochs + 1):
        base_model.train()
        head.train()
        epoch_steps = 0
        train_loss = 0.0
        for item in contracts:
            word_embeddings = _atomic_word_embeddings(base_model, tokenizer, item, device)
            pair_mask = torch.tensor([item.grid.pair_mask], dtype=torch.bool, device=device)
            labels = torch.tensor([item.grid.labels], dtype=torch.long, device=device)
            logits = head(word_embeddings, pair_mask)
            loss = nn.functional.cross_entropy(logits.reshape(-1, item.label_count), labels.reshape(-1))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            train_loss += float(loss.detach().cpu())
            epoch_steps += 1
            optimizer_steps_total += 1
        validation_metrics = evaluate_w2ner_validation(base_model, head, tokenizer, validation_contracts, device)
        row = {
            "epoch": epoch,
            "mode": mode,
            "train_loss": train_loss / max(1, epoch_steps),
            "validation_exact_f1": validation_metrics["validation_exact_f1"],
            "optimizer_steps": optimizer_steps_total,
            "internal_test_accessed": False,
        }
        with history_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
        latest_payload = checkpoint_state_payload(
            base_model=base_model,
            head=head,
            epoch=epoch,
            optimizer_steps=optimizer_steps_total,
            parameter_count=parameter_count,
            mode=mode,
        )
        torch.save(latest_payload, OUTPUT_DIR / "checkpoints" / "latest.pt")
        if float(validation_metrics["validation_exact_f1"]) >= best_metric:
            best_metric = float(validation_metrics["validation_exact_f1"])
            best_payload = latest_payload
            torch.save(best_payload, OUTPUT_DIR / "checkpoints" / "best.pt")
    if best_payload is None or latest_payload is None:
        raise RuntimeError("training finished without checkpoint payloads")
    checkpoint_hashes = {
        name: sha256_file(OUTPUT_DIR / "checkpoints" / f"{name}.pt")
        for name in ("best", "latest")
    }
    validate_checkpoint_after_save_reload(OUTPUT_DIR / "checkpoints" / "best.pt", checkpoint_hashes["best"])
    validate_checkpoint_after_save_reload(OUTPUT_DIR / "checkpoints" / "latest.pt", checkpoint_hashes["latest"])
    validation_metrics = evaluate_w2ner_validation(base_model, head, tokenizer, validation_contracts, device)
    validation_metrics.update({
        "checkpoint_hashes": checkpoint_hashes,
        "completed_epochs": epochs,
        "optimizer_steps": optimizer_steps_total,
        "parameter_count": parameter_count,
        "load_report": load_report,
        "internal_test_accessed": False,
    })
    return validation_metrics

validation_metrics = run_training(
    train_contracts,
    validation_contracts,
    mode=mode,
    epochs=FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS,
)
(OUTPUT_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "stage": "training_completed",
    "mode": mode,
    "validation_exact_f1": validation_metrics["validation_exact_f1"],
    "checkpoint_hashes": validation_metrics["checkpoint_hashes"],
    "model_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": PINNED_TOKENIZER_REVISION,
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

In [ ]:
checkpoint_hashes = dict(validation_metrics["checkpoint_hashes"])
manifest = build_e4_manifest(
    mode=mode,
    status=STATUS_FULLY_TRAINED if RUN_FULL_TRAINING else STATUS_SMOKE_EXECUTED,
    run_completed=True,
    repository_commit=RESOLVED_COMMIT,
    corpus_hashes=corpus_hashes,
    data_hashes=corpus_hashes,
    resolved_config=resolved_config,
    model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_TOKENIZER_REVISION,
    seed=SEED,
    completed_epochs=int(validation_metrics["completed_epochs"]),
    optimizer_steps=int(validation_metrics["optimizer_steps"]),
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    parameter_count=int(validation_metrics["parameter_count"]),
    checkpoint_hashes=checkpoint_hashes,
    best_metric=float(validation_metrics["validation_exact_f1"]),
    train_split_id="governed_train_sha256_" + corpus_hashes["train"],
    validation_split_id="governed_validation_sha256_" + corpus_hashes["validation"],
    safe_to_resume=True,
    initialization_source="pinned_pretrained_base" if RUN_FULL_TRAINING else "pinned_pretrained_base_bounded_smoke",
)
manifest.validate()
manifest.write(OUTPUT_DIR / "training_manifest.json")
report = validate_e4_artifact(OUTPUT_DIR, mode=mode)
print(json.dumps({
    "stage": "artifact_validation",
    "artifact_dir": str(OUTPUT_DIR),
    "manifest_sha256": report.manifest_sha256,
    "validator": report.as_dict(),
    "status": "SMOKE_EXECUTED" if RUN_SMOKE_TRAINING and not RUN_FULL_TRAINING else "FULLY_TRAINED",
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))
if not report.ok:
    raise AssertionError(report.failures)